In [1]:
# Mount Google Drive (neu chua mount)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Install Package

In [2]:
# 1. Nang cap cong cu loi
!pip install -U pip "setuptools<82.0.0" "jedi>=0.16"

# 2. Cai dat he sinh thai Unsloth & Xformers
!pip install unsloth==2026.5.2 unsloth_zoo xformers

# 3. TRL da duoc cai tu dong boi unsloth (trl==0.24.0)
# KHONG can cai them trl@main vi se gay xung dot phien ban

# Import Library

In [3]:
import os
import re
import torch
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

/tmp/ipykernel_5977/2798534343.py:5: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Load BaseModel

In [4]:
# Bật tính năng Standby của Unsloth để tiết kiệm 30%+ bộ nhớ cho RL
os.environ['UNSLOTH_VLLM_STANDBY'] = "1"

max_seq_length = 2048 # Có thể tăng lên nếu suy luận dài
lora_rank = 32 # Rank cao hơn = thông minh hơn, nhưng train chậm hơn

# Tải mô hình cơ sở
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B", # Tên model LLM
    max_seq_length = max_seq_length,  # độ dài của context length mà mô hình có thể đọc và sinh ra trong 1 chu kỳ
    load_in_4bit = True,               # quantization xuống 4-bit
    fast_inference = False,           # tắt suy luận nhanh để ổn định hệ thống
    max_lora_rank = lora_rank,        # rank của LoRA
    load_in_fp8 = False,               # tắt chế độ tải mô hình bằng định dạng độ chính xác Float8
)
# Cấu hình LoRA Adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # chọn module của model để áp dụng LoRA
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",  # dọn dẹp bộ nhớ rác sinh ra khi train giảm VRAM
    random_state = 3407,
)

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}
Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}
Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


unsloth/deepseek-r1-0528-qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


# Chat template

In [5]:
# System prompt cho EXACT 2026
system_prompt = (
    "You are an expert educational AI assistant for the EXACT 2026 competition. "
    "For logic problems: analyze premises carefully, apply formal reasoning, "
    "and derive the correct conclusion. "
    "For physics problems: identify relevant formulas, show step-by-step calculations, "
    "and provide the final numerical answer with correct units. "
    "Always think step-by-step inside <think>...</think> tags, "
    "then give your final answer inside <answer>...</answer> tags."
)


Real eos_token: '<｜end▁of▁sentence｜>'
chat_template OK, no fix needed


# Dataset

In [6]:
from datasets import load_dataset

# ====================================================================
# HUONG DAN: Upload file train.jsonl va val.jsonl len Google Drive
# vao thu muc /content/drive/MyDrive/exact2026_data/
# ====================================================================

# Load dataset EXACT 2026 tu Google Drive
DRIVE_DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Exact_2026"

train_dataset = load_dataset(
    "json",
    data_files=f"{DRIVE_DATA_DIR}/train.jsonl",
    split="train"
)

# Val dataset (optional - de danh gia)
try:
    val_dataset = load_dataset(
        "json",
        data_files=f"{DRIVE_DATA_DIR}/val.jsonl",
        split="train"
    )
    print(f"Val dataset: {len(val_dataset)} samples")
except:
    val_dataset = None
    print("No val dataset found, skipping.")

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Sample: {train_dataset[0]['conversations'][:2]}")

Val dataset: 216 samples
Train dataset: 1944 samples
Sample: [{'role': 'system', 'content': 'You are an expert educational AI assistant for the EXACT 2026 competition. For logic problems: analyze premises carefully, apply formal reasoning, and derive the correct conclusion. For physics problems: identify relevant formulas, show step-by-step calculations, and provide the final numerical answer with correct units. Always think step-by-step inside <think>...</think> tags, then give your final answer inside <answer>...</answer> tags.'}, {'role': 'user', 'content': '[LOGIC PROBLEM]\nPremises:\n1. If a student does not attend lectures, they will not practice exercises.\n2. All students practice exercises.\n\nQuestion:\nBased on the above premises, which statement can be inferred?\nA. There exists a student who has not mastered Calculus.\nB. There exists a student who does not practice exercises.\nC. All students attend lectures.\nD. There exists a student who does not perform well in exams.'

# Config

In [8]:
# Fix Unsloth hardcode eos/pad token bang cach patch file TRL
import trl, os

sft_path = os.path.join(os.path.dirname(trl.__file__), "trainer", "sft_trainer.py")
with open(sft_path, "r") as f:
    code = f.read()

# Comment out 2 doan check eos_token va pad_token trong vocab
code = code.replace(
    'raise ValueError(\n                    f"The specified `eos_token`',
    'pass  # patched\n                if False: raise ValueError(\n                    f"The specified `eos_token`'
)
code = code.replace(
    'raise ValueError(\n                    f"The specified `pad_token`',
    'pass  # patched\n                if False: raise ValueError(\n                    f"The specified `pad_token`'
)

with open(sft_path, "w") as f:
    f.write(code)

print("Patched sft_trainer.py OK")

# Reload module
import importlib
importlib.reload(trl.trainer.sft_trainer)
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir = "/content/drive/MyDrive/Exact_2026_checkpoints",
    save_steps = 50,
    save_total_limit = 3,

    learning_rate = 2e-5,
    weight_decay = 0.01,
    bf16 = False,
    fp16 = False,
    max_grad_norm = 1.0,
    warmup_steps = 10,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",

    logging_steps = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,

    num_train_epochs = 3,
    max_length = 2048,

    dataset_text_field = None,
    packing = False,

    report_to = "none",
    seed = 3407,
)

# formatting_func: chuyen conversations thanh text de SFT
def formatting_func(examples):
    texts = []
    for convs in examples["conversations"]:
        text = tokenizer.apply_chat_template(
            convs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return texts

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    formatting_func = formatting_func,
)

Patched sft_trainer.py OK
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["None"] (num_proc=6):   0%|          | 0/1944 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["None"] (num_proc=6):   0%|          | 0/216 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [9]:
import os
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

In [11]:
# ====================================================================
# CHAY TRAINING
# Neu tiep tuc tu checkpoint cu (sau khi Colab disconnect):
#   trainer.train(resume_from_checkpoint=True)
# Neu train tu dau:
#   trainer.train()
# ====================================================================

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,944 | Num Epochs = 3 | Total steps = 729
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 87,293,952 of 8,278,029,312 (1.05% trained)


KeyboardInterrupt: 

In [ ]:
# Giả sử 'model' và 'tokenizer' là cái ông đã load thành công ở bước [3]
# Thử convert sang GGUF bản 4-bit (phổ biến nhất)
model.save_pretrained_gguf(
    "test_model_gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)